In [8]:
"""
Use aimsgb to generate grain boundaries, then use 
GPUMD to relax such structures into realistic configurations.
"""
from dotenv import load_dotenv
from aimsgb import GrainBoundary, Grain
from ase.visualize import view
load_dotenv()

s_input = Grain.from_mp_id("mp-149")  # Silicon
UC_A = 2
UC_B = 2

Retrieving MaterialsDoc documents: 100%|██████████| 1/1 [00:00<00:00, 26214.40it/s]


In [13]:
# In Lortaraprasert and Shiomi 2022, they use:
# 1. Sigma 5 (2 -1 0) [0 0 1]
# 2. Sigma 9 (-1 -2 1) [1 1 0]
# 3. Sigma 9 (2 -2 1) [2 1 2]
# 4. Sigma 3 (1 -1 2) [1 1 0]
# 5. Sigma 5 (3 1 0) [3 1 0]
# 6. Sigma 13 (1 0 0) [1 0 0]
GB_LIST = [
    (5, (2, -1, 0), (0, 0, 1)),
    # (9, (-2, 2, 1), (1, 1, 0)),
    # (9, (-2, 2, 1), (2, 1, 2)),
    # (3, (1, -1, 2), (1, 1, 0)),
    # (5, (3, 1, 0), (3, 1, 0)),
    # (13, (1, 0, 0), (1, 0, 0))
]

for (sigma, miller, axis) in GB_LIST:
    gb = GrainBoundary(axis, sigma, miller, s_input, uc_a=UC_A, uc_b=UC_B)
    structure = Grain.stack_grains(gb.grain_a, gb.grain_b, direction=gb.direction)

    view(structure.to_ase_atoms())

# then convert to ASE atoms, write into GPUMD format
# use SW potential with cooling ramp to relax structure (4000K to 25K over 1275ps, simulation step 1fs)
# then a final energy relaxation using L-BFGS
# repeat this process 10 times and get the most energetically stable structure